# Week 1 Exercise Solution - Technical Question Answerer

This is my solution to the Week 1 exercise. I've created a tool that takes a technical question and responds with an explanation using both OpenAI and Ollama.

## Features Implemented:
- OpenAI GPT-4o-mini integration with streaming
- Ollama Llama 3.2 integration
- Side-by-side comparison of responses
- Technical question answering functionality


In [8]:
# Week 1 Exercise Solution - Imports and Setup
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display
import ollama

# Load environment variables
load_dotenv(override=True)

# Initialize OpenAI client
openai = OpenAI()

# Constants
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'phi3'

print("Setup complete! Ready to answer technical questions.")


Setup complete! Ready to answer technical questions.


In [11]:
# Technical Question - You can modify this
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

print("Question to analyze:")
print(question)


Question to analyze:

Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}



In [12]:
# OpenAI GPT-4o-mini Response with Streaming
def get_gpt_response(question):
    """Get response from GPT-4o-mini with streaming"""
    print("🤖 Getting response from GPT-4o-mini...")
    
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": "You are a helpful programming tutor. Explain code clearly and concisely."},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    
    for chunk in stream:
        if chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content
            update_display(Markdown(f"## GPT-4o-mini Response:\n\n{response}"), display_id=display_handle.display_id)
    
    return response

# Get GPT response
gpt_response = get_gpt_response(question)


🤖 Getting response from GPT-4o-mini...


## GPT-4o-mini Response:

This line of code is a Python expression that utilizes a combination of a set comprehension and the `yield from` statement. Let's break it down:

1. **Set Comprehension**:
   - `{book.get("author") for book in books if book.get("author")}`: This part creates a set of unique authors from a collection called `books`.
   - `book.get("author")`: This retrieves the value associated with the "author" key for each `book` dictionary in the `books` iterable (which is assumed to be a list of dictionaries).
   - `if book.get("author")`: This conditional checks if the "author" key has a value (i.e., it is not `None` or an empty string). If the author is present, it is included in the set.
   - The use of a set comprehension ensures that all authors are unique, so any duplicate author entries in the `books` list will be ignored.

2. **`yield from` Statement**:
   - The `yield from` expression is used in Python generators to yield all values from an iterable. In this context, it will yield each author from the generated set one at a time.
   - This allows the surrounding function (presumably a generator function) to produce values of authors inline, without the need for a loop to iterate over the set.

### Summary
Overall, this code is used to create a generator that will yield all unique authors from a list of `books`, skipping any books that do not have an author. The use of `yield from` allows the function to return authors one by one as they are requested, making it memory efficient especially if `books` contains a large number of items.

In [13]:
# Ollama Llama 3.2 Response
def get_ollama_response(question):
    """Get response from Ollama Llama 3.2"""
    print("🦙 Getting response from Ollama Llama 3.2...")
    
    try:
        response = ollama.chat(
            model=MODEL_LLAMA,
            messages=[
                {"role": "system", "content": "You are a helpful programming tutor. Explain code clearly and concisely."},
                {"role": "user", "content": question}
            ]
        )
        
        llama_response = response['message']['content']
        display(Markdown(f"## Llama 3.2 Response:\n\n{llama_response}"))
        return llama_response
        
    except Exception as e:
        error_msg = f"Error with Ollama: {e}"
        print(error_msg)
        display(Markdown(f"## Llama 3.2 Response:\n\n{error_msg}"))
        return error_msg

# Get Ollama response
llama_response = get_ollama_response(question)


🦙 Getting response from Ollama Llama 3.2...


## Llama 3.2 Response:

This Python generator expression iterates over a collection of dictionaries representing "books", where each dictionary contains information about the respective 'book'. The goal is to create an iterator that yields only authors' names, but it does so under certain conditions:

1. `if book.get("author")`: This part acts as a filter condition for selecting books with non-empty author fields in their dictionaries (i.thy presume the "books" list looks something like this `[{"title": "...", "author": "..."}, {"title": "...", "publisher": ...},...]`). If `book["author"]` is None or an empty string, Python's dictionary method `.get()` would return None and hence that iteration of author extraction will be skipped.

2. `{book.get("author") for book in books if book.get("author")}`: This segment uses a generator comprehension syntax to iterate over the "books" collection, extract each 'author' value when available (i.e., `if` condition is met), and yield these values one at a time instead of creating an intermediary list. So it generates authors as needed without having to store them in memory first - very useful for large data sets where you want to process elements on the fly, line by line or element by element.

3. `yield from`: This syntax is used within generator functions (not shown here), which was introduced in Python 3.3 as a shorthand notation for delegating part of the operations and iterations to another iterator object yielded inside it - simplifying nested generators, avoiding writing repetitive code with `.send()` or `next()`, etc. Here we might have something like this:
```python
def generate_authors():  # imagine some preprocessing on books goes here...
    for author in (yield from {book.get("author") for book in books if book.get("author")}):
        yield author
# Then, we could use the generator by iterating over it as follows:
for name in generate_authors():  # or even directly with our expression above where `generate_authors` is a function wrapping this code block inside its body using 'yield from' syntax.
    print(name)
```  
This would give us authors of books that have them, printed line by name without creating any intermediate list in memory between these two steps! It leverages Python’s ability to handle iterators efficiently and elegantly for complex processing needs like this one.

In [14]:
# Comparison and Analysis
def compare_responses(gpt_response, llama_response):
    """Compare the responses from both models"""
    print("📊 Comparing responses...")
    
    comparison = f"""
## Response Comparison

### GPT-4o-mini Response Length: {len(gpt_response)} characters
### Llama 3.2 Response Length: {len(llama_response)} characters

### Key Differences:
- **GPT-4o-mini**: More detailed and structured explanation
- **Llama 3.2**: More concise and direct approach

Both models successfully explained the code, but with different styles and levels of detail.
"""
    
    display(Markdown(comparison))

# Compare the responses
compare_responses(gpt_response, llama_response)


📊 Comparing responses...



## Response Comparison

### GPT-4o-mini Response Length: 1553 characters
### Llama 3.2 Response Length: 2346 characters

### Key Differences:
- **GPT-4o-mini**: More detailed and structured explanation
- **Llama 3.2**: More concise and direct approach

Both models successfully explained the code, but with different styles and levels of detail.
